# Run LLMs Locally on CPU — Complete Tutorial

This notebook walks through the full pipeline to run open-source LLMs on CPU:
1. Choose a model
2. Download it from Hugging Face
3. Convert + quantize with llama.cpp ()
4. Run inference and benchmark with  directly

## Pipeline Architecture



---

## Windows users: Set up WSL first

This tutorial runs on **Linux**. On Windows, use WSL (Windows Subsystem for Linux) for native performance.

**1. Install WSL** (PowerShell as Administrator):

Restart your machine. Ubuntu is installed by default.

**2. Connect VSCode to WSL:**
- Install the **WSL** extension in VSCode (by Microsoft)
- Click the green  button (bottom-left) -> **Connect to WSL**

From this point on, everything runs inside Ubuntu.

---

## Install dependencies

> **Critical:** Without the  flags below,  uses generic SSE2 -> **~0.2 tok/s instead of 10-20 tok/s**.



> Use the  wheel for Option B, not  -- the  wheel is SSE2 only and ~4-10x slower.

> **Source:** [llama-cpp-python GitHub](https://github.com/abetlen/llama-cpp-python)


## 1. Choose Your Model


| Model | Active size | Architecture | Strengths | HuggingFace ID |
|---|---|---|---|---|
| Qwen2.5-Coder-1.5B-Instruct | 1.5B | Dense | Ultra-light, coding + math, 30+ tok/s on AVX2 | `Qwen/Qwen2.5-Coder-1.5B-Instruct` |
| Ministral-3B-Instruct | 3B | Dense | Best efficiency/perf ratio, edge-optimized, Apache 2.0 | `mistralai/Ministral-3-3B-Instruct-2512` |
| Nanbeige4.1-3B | 3B | Dense | Strong agentic reasoning, 500+ tool calls | `Nanbeige/Nanbeige4.1-3B` |
| Phi-4-mini-instruct | 3.8B | Dense | Excellent reasoning, matches 7B quality | `microsoft/Phi-4-mini-instruct` |
| **Gemma 4 E4B** | **~4B active** | **MoE** | **Quality above its size, 128K ctx** | `google/gemma-4-E4B-it` |
| **Qwen3-4B-Thinking** | **4B** | **Dense + Thinking** | **Explicit reasoning chain, 262K ctx** | `Qwen/Qwen3-4B-Thinking-2507` |
| Qwen2.5-7B-Instruct | 7B | Dense | Best overall for general tasks, strongest tool use | `Qwen/Qwen2.5-7B-Instruct` |
| rnj-1-instruct | 8B | Dense | Code/STEM + agentic, strong tool-calling, 32K ctx | `EssentialAI/rnj-1-instruct` |


## 2. Configuration

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path(".").resolve()))
from cpu_optimizer import cpu_optimizer
from utils import get_system_info, check_llama_simd, print_simd_fix, load_model, find_gguf
from huggingface_hub import snapshot_download

# ── Paths ─────────────────────────────────────────────────────────────────────────
BASE_DIR  = Path("./models")
FULL_DIR  = BASE_DIR / "full"
GGUF_DIR  = BASE_DIR / "gguf"
LLAMA_DIR = BASE_DIR / "llama.cpp"

# ── Quantization profiles ──────────────────────────────────────────────────────
QUANT_PROFILES = {
    "precision": "Q6_K",    # best quality, more RAM
    "balanced":  "Q4_K_M",  # good tradeoff — default
    "speed":     "Q3_K_M",  # fastest, less RAM
}

# ── Edit these two lines ────────────────────────────────────────────────────────────────
MODEL_NAME    = "mistralai/Ministral-3B-Instruct-2412"  # HuggingFace model ID
QUANT_PROFILE = "balanced"                              # precision | balanced | speed

QUANT_TYPE = QUANT_PROFILES[QUANT_PROFILE]
for d in [FULL_DIR, GGUF_DIR]: d.mkdir(parents=True, exist_ok=True)

print(f"Model   : {MODEL_NAME}")
print(f"Profile : {QUANT_PROFILE}  ->  {QUANT_TYPE}")


c:\Users\a871169\OneDrive - ATOS\Bureau\geospatiale\medium\model_on_cpu\llm_to_cpu\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 3. System Info

In [ ]:
from utils import get_system_info

info = get_system_info()
print(f"CPU    : {info['cpu_brand']}")
print(f"Cores  : {info['physical_cores']}P / {info['logical_cores']}L")
print(f"RAM    : {info['ram_total_gb']} GB total  |  {info['ram_avail_gb']} GB available")
print(f"SIMD   : {info['simd_flags'] or ['none detected']}")


CPU        : 12th Gen Intel(R) Core(TM) i7-1265U
Cores      : 10P / 12L
RAM        : 13.2 GB available / 31.7 GB total
SIMD       : ['avx2']


In [ ]:
# SIMD build check

from utils import check_llama_simd, print_simd_fix

cpu_has, build_has, missing = check_llama_simd()

print(f"CPU   : AVX2={cpu_has['avx2']}  AVX-512={cpu_has['avx512f']}  AMX={cpu_has['amx_tile']}")
print(f"Build : AVX2={build_has['avx2']}  AVX-512={build_has['avx512f']}  AMX={build_has['amx_tile']}")

if missing:
    print_simd_fix(missing)
else:
    print("\n✅ llama-cpp-python compiled with the correct SIMD flags.")
    print("   Source: https://github.com/abetlen/llama-cpp-python")


CPU   : AVX2=True  AVX-512=False  AMX=False
Build : AVX2=True  AVX-512=True  AMX=False

✅ llama-cpp-python compiled with the correct SIMD flags.
   Source: https://github.com/abetlen/llama-cpp-python


## 4. Download the Model

In [ ]:

dest = FULL_DIR / MODEL_NAME

if dest.exists() and any(dest.iterdir()):
    print(f"Already downloaded: {dest}")
else:
    print(f"Downloading {MODEL_NAME} ...")
    snapshot_download(
        repo_id=MODEL_NAME,
        local_dir=str(dest),
        ignore_patterns=["*.msgpack", "*.h5", "flax_model*", "tf_model*"],
    )
    print(f"Saved to: {dest}")

Already downloaded: models\full\mistralai\Ministral-3-3B-Instruct-2512


## 5. Quantize with `cpu_optimizer`

The `cpu_optimizer` class (from `cpu_optimizer.py`) automates the full conversion pipeline:
- Clones & compiles `llama.cpp` with the right SIMD flags for your CPU
- Converts the HuggingFace model to GGUF F16
- Quantizes to the target format
- Cleans up the intermediate F16 file

### GGUF Formats: Speed, RAM, and Actual Quality

> **Note:** K-quant formats (Q4_K_M, Q5_K_M, Q6_K) are not exactly the advertised bit-width.
> They use hierarchical super-blocks where critical layers (attention, FFW) are promoted to Q6_K.
> Q4_K_M is actually ~4.9 bits/weight on an 8B model, not 4.0.
>
> Sources: [llama.cpp quantize README](https://github.com/ggml-org/llama.cpp/blob/master/tools/quantize/README.md) · [arXiv 2601.14277 — perplexity benchmarks](https://arxiv.org/abs/2601.14277)

| Profile | Type | Actual bits (8B) | RAM (3B) | RAM (7B) | Perplexity (WikiText-2) | Use case |
|---------|------|-----------------|----------|----------|------------------------|----------|
| precision | Q6_K | 6.56 | ~2.9 GB | ~6.1 GB | ~7.35 | Production, max quality |
| **balanced** | **Q4_K_M** | **4.89** | **~2.0 GB** | **~4.6 GB** | **7.56** | **Recommended — best trade-off** |
| speed | Q3_K_M | ~3.5 | ~1.6 GB | ~3.7 GB | ~8.0 | Very tight RAM |
| — | Q8_0 | 8.50 | ~3.3 GB | ~7.9 GB | 7.33 | Near-identical to FP16 |
| — | F16 | 16.00 | ~6.5 GB | ~14.9 GB | 7.32 (baseline) | Reference only, not practical on CPU |

**Takeaway:** Q4_K_M loses only 0.24 perplexity points vs FP16 — imperceptible in practice. Q3_K_M starts to degrade response coherence.


In [ ]:

optimizer = cpu_optimizer(
    model_name=MODEL_NAME,
    input_path=str(FULL_DIR) + "/",
    output_path=str(GGUF_DIR) + "/",
    quant_type=QUANT_TYPE,
    quantization_verbose=False,
)
optimizer.llama_dir = LLAMA_DIR

model_folder = MODEL_NAME.split("/", 1)[1] if "/" in MODEL_NAME else MODEL_NAME
f16_path   = GGUF_DIR / f"{model_folder}-f16.gguf"
quant_path = GGUF_DIR / f"{model_folder}-{QUANT_TYPE}.gguf"

# ── Case 1: already quantized ──────────────────────────────────────────────
if quant_path.exists():
    print(f"Already quantized: {quant_path.name} ({quant_path.stat().st_size/1024**3:.2f} GB)")
    gguf_path  = quant_path
    quant_used = QUANT_TYPE
    success    = True

# ── Case 2: F16 exists, skip conversion ───────────────────────────────────
elif f16_path.exists():
    print(f"F16 found ({f16_path.stat().st_size/1024**3:.1f} GB) — running quantization only")
    optimizer.gguf_f16 = f16_path
    optimizer._auto_detect_model_params()
    optimizer._select_quant_type()
    optimizer._setup_directories()

    cpu_info = optimizer._detect_cpu_capabilities()
    build_flags, build_threads = cpu_info[0], cpu_info[1]
    quant_threads = cpu_info[2] if len(cpu_info) > 2 else build_threads

    optimizer._clone_llama_cpp()
    optimizer._build_llama_cpp(build_flags, build_threads)

    quantize_bin = optimizer._get_quantize_bin()

    cmd = f'"{quantize_bin}" "{f16_path}" "{optimizer.gguf_quantized}" {QUANT_TYPE} {quant_threads}'
    subprocess.run(cmd, shell=True)  # don't use check=True — llama-quantize exits 1 on Windows even on success

    if optimizer.gguf_quantized.exists():
        optimizer._cleanup()
        gguf_path  = optimizer.gguf_quantized
        quant_used = QUANT_TYPE
        success    = True
    else:
        print("Quantization failed — output file not found")
        success = False

# ── Case 3: full pipeline ─────────────────────────────────────────────────
else:
    success, quant_used = optimizer.main_optimize()
    if success:
        gguf_path = optimizer.gguf_quantized

if success:
    print(f"\nGGUF : {gguf_path}")
    print(f"Size : {gguf_path.stat().st_size / 1024**3:.2f} GB")


🚀 Optimising model for CPU: mistralai/Ministral-3-3B-Instruct-2512

🔍 Auto-detecting model parameters from config.json...
   Size extracted from model name: 3.0B
✅ Detected parameters:
   Parameters : ~3.0B
   Layers     : 40
   Heads      : 8
   Head dim   : 640
   Max context: 131,072 tokens (131k)

🎯 Quantisation: Q3_K_M
⚠️  Aggressive quantisation: quality reduced (more errors / hallucinations possible).
   Recommendation: use 'balanced' or 'precision' profile if RAM allows.
📂 Source model : Ministral-3-3B-Instruct-2512
📂 GGUF output  : gguf
📂 llama.cpp    : llama.cpp

💾 Total RAM    : 31.7 GB
💾 Available RAM: 13.2 GB

📊 Model 3.0B Q3_K_M
  Estimated size    : 1.4 GB
  RAM after model   : ~11.8 GB
  KV cache estimate : 10.16 GB
  Context           : 13312 tokens (13k)



🖥️  CPU: 12th Gen Intel(R) Core(TM) i7-1265U
  AVX2        : ✓
  AVX-512     : ✗
  AVX-512VNNI : ✗
  AMX         : ✗

  Physical cores : 10
  Logical cores  : 12

  Build flags   : -DGGML_NATIVE=ON -DGGML_OPENMP=ON -DGGML_AVX=ON -DGGML_AVX2=ON
  LTO           : ON
  Build threads : 8
  Quant threads : 12

🔄 Updating llama.cpp...

>>> git fetch --depth=1 origin

>>> git reset --hard FETCH_HEAD
HEAD is now at 0988acc [SYCL] Add Q8_0 reorder optimization (~3x tg speedup on Intel Arc) (#21527)

   Already up to date (0988accf).

✅ llama.cpp already built — skipping compilation.
  Disk space: 396.5 GB available (required: ~5.6 GB) ✓

🔄 Converting HF → GGUF F16...

>>> "c:\Users\a871169\OneDrive - ATOS\Bureau\geospatiale\medium\model_on_cpu\llm_to_cpu\.venv\Scripts\python.exe" convert_hf_to_gguf.py "C:\Users\a871169\OneDrive - ATOS\Bureau\geospatiale\medium\model_on_cpu\llm_to_cpu\models\full\mistralai\Ministral-3-3B-Instruct-2512" --outtype f16 --outfile "C:\Users\a871169\OneDrive - ATOS\B

## 6. Load the Model and Run Inference

We use `llama-cpp-python` directly — no extra wrapper needed.

In [ ]:
from utils import load_model, find_gguf

gguf_path = find_gguf(MODEL_NAME, GGUF_DIR)
n_ctx_hint = getattr(optimizer, 'ctx', None) if 'optimizer' in dir() else None

model, load_info = load_model(gguf_path, n_ctx=n_ctx_hint)


  Context (cpu_optimizer): 13312 tokens
  CPU: 10 physical cores  →  inference: 9 threads  |  batch: 10 threads
  Context: 13312 tokens
  Threads — inference: 9  |  batch: 10
  KV cache Q8_0 enabled (ctx=13312 > 4096) — ~50% RAM saving


llama_context: setting new yarn_attn_factor = 1.0000 (mscale == 1.0, mscale_all_dim = 1.0)
llama_context: n_ctx_seq (13312) < n_ctx_train (262144) -- the full capacity of the model will not be utilized



Model loaded in 5.6s  |  +1.9 GB RAM  |  ctx=13312 tokens


## 7. Generate Text + Measure Performance

In [ ]:
from utils import generate as _generate, warmup

SYSTEM_PROMPT = (
    "You are a helpful, concise assistant. "
    "Answer clearly and directly. "
    "If the question is about code, provide working examples."
)

def generate(prompt: str, max_tokens: int = 150, stream: bool = True) -> dict:
    return _generate(model=model, prompt=prompt,
                     system_prompt=SYSTEM_PROMPT,
                     max_tokens=max_tokens, stream=stream)

warmup(model)


Warming up... ready.


In [ ]:
result = generate(
    prompt="Explain what a Large Language Model is in 3 sentences.",
    max_tokens=150,
    stream=True,
)
print(f"
{result['tok_s']} tok/s  |  {result['tokens']} tokens  |  "
      f"gen {result['generation_time_s']}s  |  CPU {result['avg_cpu_pct']}%")


A Large Language Model (LLM) is an advanced type of artificial intelligence that uses deep learning to analyze and generate human-like text based on vast amounts of data. It leverages transformer architectures—like those in models like me—to understand context, intent, and language patterns with high accuracy. LLMs power tools like chatbots, content creation, and even translation services by predicting and generating coherent responses.

1.1 tok/s  |  82 tokens  |  75.27s  |  CPU 100.0%


## 8. Benchmark Multiple Prompts

In [ ]:
prompts = [
    "What is quantization in machine learning?",
    "Write a Python function to sort a list of dictionaries by a key.",
    "What are the main advantages of running LLMs locally?",
]

rows = []
for p in prompts:
    print(f"\n── {p[:60]}")
    r = generate(p, max_tokens=100, stream=True)
    rows.append(r)

print(f"\n{'Prompt':<55} {'tok/s':>6} {'tokens':>7} {'time':>6}")
print("-" * 78)
for p, r in zip(prompts, rows):
    print(f"{p[:54]:<55} {r['tok_s']:>6} {r['tokens']:>7} {r['time_s']:>5}s")

avg = sum(r['tok_s'] for r in rows) / len(rows)
rating = "Excellent" if avg > 20 else "Good" if avg > 10 else "Acceptable" if avg > 5 else "Slow"
print(f"\nAverage: {avg:.1f} tok/s  →  {rating}")


── What is quantization in machine learning?
Quantization in machine learning refers to the process of reducing the precision of numerical values used during model training, inference, or deployment while maintaining the overall performance and accuracy. It is commonly applied in deep neural networks (DNNs) to optimize memory usage, computational efficiency, and power consumption.

### Key Concepts:
1. **Bit-width Reduction**:
   - Original data or weights are stored with high precision

 (e

KeyboardInterrupt: 

## Summary

| Step | Tool | Key setting |
|------|------|-------------|
| Install | `pip install llama-cpp-python` | `CMAKE_ARGS="-DGGML_BLAS=ON -DGGML_BLAS_VENDOR=OpenBLAS -DGGML_AVX2=ON"` |
| SIMD check | `check_llama_simd()` (utils.py) | AVX2 missing → 50× slower |
| Download | `huggingface_hub.snapshot_download` | `ignore_patterns` to skip TF/Flax weights |
| Quantize | `cpu_optimizer` | `QUANT_PROFILE = "balanced"` (Q4_K_M recommended) |
| Load | `load_model()` (utils.py) | physical cores, `flash_attn=True`, KV cache Q8_0 if ctx > 4096 |
| Infer | `generate()` (utils.py) | `format_prompt()` auto-detects template from model family |
| Benchmark | `generate()` | Always warmup first — first call is always slower |

---

### Expected performance (Q4_K_M, llama-cpp-python built with OpenBLAS + AVX2)

> ⚠️ These numbers assume a correct installation with `CMAKE_ARGS`. Without SIMD flags, expect 0.2–2 tok/s.

| Architecture | 1.5B model | 3B model | 7B model |
|---|---|---|---|
| x86 AVX2 (Intel/AMD consumer) | 20–35 tok/s | 12–20 tok/s | 5–10 tok/s |
| x86 AVX-512 (Xeon, some Core) | 35–55 tok/s | 20–35 tok/s | 10–18 tok/s |
| x86 AMX (Xeon 4th gen+, SPR/EMR) | 60+ tok/s | 35–55 tok/s | 20–35 tok/s |
| ARM (Apple M-series, Ampere) | 25–45 tok/s | 15–28 tok/s | 8–15 tok/s |

Source: [llama.cpp quantize README](https://github.com/ggml-org/llama.cpp/blob/master/tools/quantize/README.md)

---

### Quantization quality (perplexity on WikiText-2, 8B model)

| Format | Actual bits | Perplexity | vs FP16 |
|--------|------------|-----------|--------|
| F16 (baseline) | 16.00 | 7.32 | — |
| Q8_0 | 8.50 | 7.33 | +0.01 ✅ |
| Q6_K | 6.56 | ~7.35 | +0.03 ✅ |
| **Q4_K_M** | **4.89** | **7.56** | **+0.24 ✅** |
| Q3_K_M | ~3.5 | ~8.0 | +0.68 ⚠️ |

Source: [arXiv 2601.14277 (2025)](https://arxiv.org/abs/2601.14277)

---

### Ollama: CPU optimisation (alternative to llama-cpp-python)

[Ollama](https://ollama.com/download) provides a native installer for Windows/Linux/macOS (no WSL2 required on Windows).

```bash
# Linux/macOS
export OLLAMA_NUM_THREADS=$(nproc)
export OLLAMA_FLASH_ATTENTION=1
export OLLAMA_KV_CACHE_TYPE=q8_0
ollama serve

# Windows PowerShell
$env:OLLAMA_NUM_THREADS = [Environment]::ProcessorCount
$env:OLLAMA_FLASH_ATTENTION = "1"
$env:OLLAMA_KV_CACHE_TYPE = "q8_0"
ollama serve
```

---

### If you get < 2 tok/s — re-run the SIMD diagnostic cell (Section 3)

---

## References

1. **llama.cpp quantize README** (GGUF formats, actual bits/weight, benchmarks)  
   https://github.com/ggml-org/llama.cpp/blob/master/tools/quantize/README.md

2. **"Which Quantization Should I Use?"** — arXiv 2601.14277 (2025)  
   WikiText-2 perplexity data for all GGUF formats  
   https://arxiv.org/abs/2601.14277

3. **llama.cpp Performance Tuning** (thread count rule: physical cores, HT harmful)  
   https://www.mintlify.com/ggml-org/llama.cpp/advanced/performance-tuning

4. **llama-cpp-python GitHub** (installation, CMAKE_ARGS, BLAS backends)  
   https://github.com/abetlen/llama-cpp-python

5. **llama.cpp P-core/E-core Intel discussion** (Intel hybrid 12th–14th gen)  
   https://github.com/ggml-org/llama.cpp/discussions/572

6. **Intel guide: llama.cpp + iGPU**  
   https://www.intel.com/content/www/us/en/developer/articles/technical/run-llms-on-gpus-using-llama-cpp.html

7. **Ollama native Windows app** (no WSL2 needed)  
   https://ollama.com/blog/windows-preview

8. **CPU vs iGPU shared VRAM benchmark** (partial -ngl harmful for integrated GPUs)  
   https://dev.to/maximsaplin/llamacpp-cpu-vs-gpu-shared-vram-and-inference-speed-3jpl
